In [1]:
import torch
import torch.nn as nn

In [2]:
# input & output initialization
x = torch.tensor([
    [1, 2, 3, 4, 5],
    [2, 3, 4, 5, 6],
    [3, 4, 5, 6, 7],
    [1, 3, 5, 7, 9],
    [0, 2, 4, 6, 8],
    [4, 5, 6, 7, 8],
    [5, 4, 3, 2, 1],
    [6, 5, 4, 3, 2],
    [9, 7, 5, 3, 1],
    [3, 1, 4, 1, 5],
    [2, 7, 1, 8, 2],
    [4, 4, 4, 4, 4],
], dtype=torch.float32)
# 1 if increasing, 0 if decreasing (mixed or flat also count as 0)
y = torch.tensor([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0], dtype=torch.float32).view(-1, 1)
neurons = 4

In [3]:
# model declaration
class MultiLayerPerceptron(nn.Module):
    def __init__(self, input_size, output_size, hidden_size):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_features=input_size, out_features=hidden_size),
            nn.ReLU(),
            nn.Linear(in_features=hidden_size, out_features=output_size),
        )

    def forward(self, input):
        return self.head(input)

In [4]:
# model initialization
model = MultiLayerPerceptron(input_size=x.shape[-1], output_size=y.shape[-1], hidden_size=neurons)
for name, param in model.named_parameters():
    print(f"{name}: {param.data}")
# optimizer initialization
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

head.0.weight: tensor([[ 0.1722, -0.0832, -0.0496, -0.1478, -0.4161],
        [ 0.1246,  0.3690,  0.3567, -0.1295,  0.2642],
        [-0.4199,  0.4347, -0.3756,  0.2898, -0.1451],
        [-0.3100, -0.3644,  0.0725,  0.4338,  0.3005]])
head.0.bias: tensor([ 0.3602,  0.3969, -0.1559,  0.2870])
head.2.weight: tensor([[-0.1745,  0.1560,  0.3847,  0.4329]])
head.2.bias: tensor([-0.2118])


In [5]:
# training
epochs = 500
for epoch in range(epochs):
    # forward pass
    z = model(x)
    # calculate loss
    loss = nn.BCEWithLogitsLoss()(input=z, target=y)
    # backward pass
    optimizer.zero_grad()
    loss.backward()
    if epoch % 50 == 0:
        print(f"epoch {epoch} loss: {loss.item()}")
        for name, param in model.named_parameters():
            print(f"{name}.grad: {param.grad}")
    # update parameters
    optimizer.step()

epoch 0 loss: 0.706221878528595
head.0.weight.grad: tensor([[-0.1284, -0.1008, -0.0732, -0.0456, -0.0180],
        [ 0.2321,  0.2283,  0.1415,  0.1519,  0.0743],
        [ 0.0579,  0.2027,  0.0290,  0.2317,  0.0579],
        [ 0.1911,  0.2713,  0.1213,  0.2410,  0.1166]])
head.0.bias.grad: tensor([-0.0180,  0.0447,  0.0290,  0.0568])
head.2.weight.grad: tensor([[0.0134, 1.1625, 0.2786, 0.0152]])
head.2.bias.grad: tensor([0.2865])
epoch 50 loss: 0.12137734144926071
head.0.weight.grad: tensor([[-0.0547, -0.0195, -0.0167,  0.0267,  0.0042],
        [-0.0021,  0.0022, -0.0004,  0.0053,  0.0012],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0593,  0.0153,  0.0137, -0.0410, -0.0095]])
head.0.bias.grad: tensor([-0.0160, -0.0004,  0.0000,  0.0186])
head.2.weight.grad: tensor([[ 0.0839,  0.0234,  0.0000, -0.0809]])
head.2.bias.grad: tensor([0.0245])
epoch 100 loss: 0.04470757022500038
head.0.weight.grad: tensor([[-0.0194, -0.0093,  0.0013,  0.0127,  0.0081],
        [-0.00

In [6]:
# prediction
with torch.inference_mode():
    z = model(x)
# activation function (value > 0 is 1, else 0)
y_pred = (z > 0).float()
print(f"y_pred: {y_pred.view(-1)}")
print(f"y:      {y.view(-1)}")

y_pred: tensor([1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.])
y:      tensor([1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.])
